# Week 52, Day 3: Review III: senses, movement, plasticity and memory

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: Hebbian learning finds the principal component

Apply Oja's rule, w += eta * y * (x - y * w) with y = w . x, to 5,000 inputs drawn from a correlated Gaussian, and compare the result with the top eigenvector of the input covariance. Run plain Hebbian learning, w += eta * y * x, for comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(42)
C = np.array([[3.0, 1.2], [1.2, 1.0]])
xs = r.multivariate_normal([0, 0], C, 5000)
evals, evecs = np.linalg.eigh(C); pc1 = evecs[:, -1]
eta = 0.005
w = r.normal(0, 0.1, 2); w_hebb = w.copy()
for x in xs:
    y = w @ x
    w = ...
    y_h = w_hebb @ x
    w_hebb = ...
print(f"Oja: w = {np.round(w, 3)}, length {np.linalg.norm(w):.3f}; top eigenvector {np.round(pc1, 3)}")
print(f"alignment |cos| = {abs(w @ pc1) / np.linalg.norm(w):.4f}; plain Hebb length {np.linalg.norm(w_hebb):.3g}")

In [ ]:
import numpy as np
def test_norm():
    "the weight vector converges to unit length"
    assert abs(np.linalg.norm(w) - 1) < 0.05
def test_pc():
    "the weight vector is aligned with the top eigenvector of the input covariance"
    assert abs(np.dot(w, pc1)) / np.linalg.norm(w) > 0.99
def test_hebb():
    "plain Hebbian learning on the same inputs grows without bound"
    assert np.linalg.norm(w_hebb) > 100

test_norm()
test_pc()
test_hebb()
print("All checks passed")

### Exercise: Problem 2: spontaneous recovery from two processes

Simulate the two-rate model: on each trial the error is e = p - (xf + xs), then xf = Af xf + Bf e and xs = As xs + Bs e. Use 200 trials with p = 1, 5 with p = -1, then 100 error-clamp trials in which the error is forced to zero.

In [ ]:
Af, Bf, As, Bs = 0.59, 0.21, 0.992, 0.02          # Smith, Ghazizadeh and Shadmehr (2006)
p = np.r_[np.ones(200), -np.ones(5), np.zeros(100)]
clamp = np.r_[np.zeros(205, bool), np.ones(100, bool)]
xf, xs = np.zeros(306), np.zeros(306)
for n in range(305):
    e = 0.0 if clamp[n] else ...
    xf[n + 1] = ...
    xs[n + 1] = ...
x_net = (xf + xs)[1:]; xf, xs = xf[1:], xs[1:]
print(f"end of learning: {x_net[199]:.2f}; after the reverse phase: {x_net[204]:.2f}; peak in the clamp: {x_net[205:].max():.2f} at trial {205 + np.argmax(x_net[205:])}")
plt.plot(x_net, label="net"); plt.plot(xf, label="fast"); plt.plot(xs, label="slow"); plt.axvline(205, color="grey")
plt.xlabel("trial"); plt.ylabel("adaptation"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_learn():
    "net adaptation reaches 70% of the perturbation by the end of the first phase"
    assert x_net[199] > 0.7
def test_washout():
    "after the reverse phase net adaptation is near zero"
    assert abs(x_net[204]) < 0.15
def test_rebound():
    "in the error clamp, net adaptation rebounds above 0.2 while the fast process decays"
    assert x_net[205:].max() > 0.2 and abs(xf[-1]) < 0.05

test_learn()
test_washout()
test_rebound()
print("All checks passed")

### Exercise: Problem 3: how many memories?

Complete recall(P): store P random patterns of N = 200 units with the Hebbian rule (zero diagonal), corrupt each by flipping 10% of its units, run 20 synchronous updates x = sign(Wx) for all cues at once, and return the fraction whose overlap with the stored pattern exceeds 0.95.

In [ ]:
N = 200

def recall(P, seed=0):
    g = np.random.default_rng(seed)
    xi = g.choice([-1.0, 1.0], (P, N))
    W = ...
    np.fill_diagonal(W, 0)
    cues = xi.copy()
    flip = g.random((P, N)) < 0.1
    cues[flip] *= -1
    X = cues.T                                   # N x P, one cue per column
    for _ in range(20):
        X = ...
    overlap = np.sum(X.T * xi, axis=1) / N
    return np.mean(overlap > 0.95)

Ps = np.array([5, 10, 15, 20, 25, 30, 35, 40, 50, 60])
fr = np.array([recall(P) for P in Ps])
capacity = Ps[fr >= 0.9].max()
for P, f in zip(Ps, fr):
    print(f"P = {P:2d} (P/N = {P / N:.3f}): {f:.0%} recalled")
print(f"largest P with at least 90% recall: {capacity} (P/N = {capacity / N:.3f}; theory about 0.14)")

In [ ]:
import numpy as np
def test_recall():
    "recall returns the fraction of patterns recovered with overlap above 0.95"
    assert 0 <= recall(10) <= 1 and recall(5) > 0.9
def test_capacity():
    "capacity is the largest P tested with at least 90% recall, between 15 and 35"
    assert 15 <= capacity <= 35 and recall(60) < 0.5

test_recall()
test_capacity()
print("All checks passed")